In [1]:
import sys
sys.path.append("D:/metal-flow/src/metal_flow")

In [ ]:
from qiskit_metal import Dict, view
from functions import create_design
import os
os.environ["QISKIT_METAL_HEADLESS"] = "1"


FOUR_QUBIT_DESIGN_DICT = Dict(
    # Define the overall dimensions of the planar silicon chip
    chip_size = Dict(
        size_x = '1mm',
        size_y = '10mm',
        size_z = '-280um',
        centre_x = '0.5mm',
        centre_y = '5mm'
    ),

    # Define standard Coplanar Waveguide (CPW) parameters used across the chip
    cpw_dims = Dict(
        width = '10 um',
        gap = '6 um'
    ),

    

    # Substrate and metal film thickness for EM simulations (Ansys Q3D/HFSS)
    physical_params = Dict(
        substrate_thickness = 280e-6,
        film_thickness = 200e-9
    ),

    # Locations and orientations of the 4 wirebond launchpads for I/O
    launchpad_options = Dict(
        p0 = Dict(pos_x='0.5mm' , pos_y='8500um', orientation='-90', lead_length='50 um', pad_width='80 um', pad_height='80 um'),
        #p1 = Dict(pos_x='7250um', pos_y='8500um', orientation='-90', lead_length='50 um', pad_width='80 um', pad_height='80 um'),
        p2 = Dict(pos_x='0.5mm' , pos_y='2000um' , orientation='90', lead_length='50 um', pad_width='80 um', pad_height='80 um'),
        #p3 = Dict(pos_x='7250um', pos_y='2000um' , orientation='90', lead_length='50 um', pad_width='80 um', pad_height='80 um')
    ),


    cpw_default_options = Dict(chip='main', hfss_wire_bonds=True),


    feedline_connections = [Dict(start_pin=Dict(component='p0', pin='tie'), end_pin=Dict(component='p2', pin='tie')) 
                            
                            #Dict(start_pin=Dict(component='ctl0', pin='prime_start'), end_pin=Dict(component='ctl2', pin='prime_end')),
                            
                            #Dict(start_pin=Dict(component='ctl2', pin='prime_start'), end_pin=Dict(component='p2', pin='tie')),
                            
                            #Dict(start_pin=Dict(component='p1', pin='tie'), end_pin=Dict(component='ctl1', pin='prime_start')), 
                            
                            #Dict(start_pin=Dict(component='ctl1', pin='prime_end'), end_pin=Dict(component='ctl3', pin='prime_start')),
                            
                            #Dict(start_pin=Dict(component='ctl3', pin='prime_end'), end_pin=Dict(component='p3', pin='tie'))

    ]
)

if __name__ == '__main__':
    # from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
    # from qiskit_metal import designs
    # design = designs.DesignPlanar()
    # ctl = CoupledLineTee(design)

    # print(ctl.pin_names)
    design = create_design(FOUR_QUBIT_DESIGN_DICT)
    for name, component in design.components.items():
        # Check if the component has the wire bond option
        if 'hfss_wire_bonds' in component.options:
            component.options.hfss_wire_bonds = True
            
    # 3. Rebuild the geometry to apply the changes
    design.rebuild()
    view(design).savefig("design.png")

Generating Launchpads...
Generating CoupledLineTee
Connecting...
Generating Qubits...
Generating Readout Resonators...
Generating Couplers...


In [17]:
hfss_renderer = design.renderers.hfss
hfss_renderer.start()

INFO 11:26AM [connect_project]: Connecting to Ansys Desktop API...
INFO 11:26AM [load_ansys_project]: 	Opened Ansys App
INFO 11:26AM [load_ansys_project]: 	Opened Ansys Desktop v2023.1.0
INFO 11:26AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/Dell/Documents/Ansoft/
	Project:   Project6
INFO 11:26AM [connect_design]: 	Opened active design
	Design:    EigenSim [Solution type: Eigenmode]
INFO 11:26AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 11:26AM [connect]: 	Connected to project "Project6" and design "EigenSim" 😀 



True

In [10]:
hfss_eigenmode = hfss_renderer.new_ansys_design("EigenSim",'eigenmode')

INFO 11:09AM [connect_design]: 	Opened active design
	Design:    EigenSim [Solution type: Eigenmode]
WARNING 11:09AM [connect_setup]: 	No design setup detected.
WARNING 11:09AM [connect_setup]: 	Creating eigenmode default setup.
INFO 11:09AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


In [18]:
hfss_renderer.clean_active_design()
hfss_renderer.render_design(selection = [], open_pins=[], port_list = [('p0','in',50),('p2','in',50)],
                            box_plus_buffer = False)

In [19]:
hfss_renderer.add_eigenmode_setup(name = "EigenSetup", n_modes = '2', max_passes = 12, delta_f = 0.5, min_converged = 2)

In [20]:
from qiskit_metal.analyses.simulation import ScatteringImpedanceSim
Driven_mode = ScatteringImpedanceSim(design,"hfss")
hfss_DrivenMode = Driven_mode.renderer

In [21]:
hfss_DrivenMode.activate_ansys_design("DrivenSim",'drivenmodal')

11:33AM 30s WARNING [activate_ansys_design]: The design_name=DrivenSim was not in active project.  Designs in active project are: 
['EigenSim'].  A new design will be added to the project.  
INFO 11:33AM [connect_design]: 	Opened active design
	Design:    DrivenSim [Solution type: DrivenModal]
WARNING 11:33AM [connect_setup]: 	No design setup detected.
WARNING 11:33AM [connect_setup]: 	Creating driven modal default setup.
INFO 11:33AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssDMSetup'>)


In [22]:
hfss_DrivenMode.render_design(selection=[], open_pins=[], port_list = [('p0','in',50),('p2','in',50)],
                                 box_plus_buffer = False)